In [1]:
pip install transformers datasets torch pandas scikit-learn


In [6]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Mock Dataset
data = {
    'text': [
        "Отличный товар, мне очень нравится!",
        "Ужасное качество, никому не рекомендую.",
        "Быстрая доставка и хороший сервис.",
        "Сломалось на следующий день после покупки.",
        "Нормальный фильм, но на один раз."
    ],
    'label': [1, 0, 1, 0, 1]
}
df = pd.DataFrame(data)

# 3. Splitting & Tokenizer
MODEL_NAME = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# 4. Tokenization Function
def tokenize_maps(df_slice):
    encodings = tokenizer(list(df_slice['text']), truncation=True, padding=True, max_length=128)
    encodings['labels'] = list(df_slice['label'])
    return encodings

train_encodings = tokenize_maps(train_df)
val_encodings = tokenize_maps(val_df)

# 5. Dataset Wrapper for Trainer
class DatasetObjects(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
    def __len__(self):
        return len(self.encodings['labels'])

train_dataset = DatasetObjects(train_encodings)
val_dataset = DatasetObjects(val_encodings)

# 6. Initialize Model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

# 7. Define Trainer Arguments (Error causing line removed)
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=0,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=1,
    report_to="none"  # Extra warning/logs block karne ke liye
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# 8. Run Training
trainer.train()

# 9. Quick Manual Test
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=1).item()
    return "Positive" if prediction == 1 else "Negative"

test_text = "Этот продукт превзошел все мои ожидания!"
print(f"\nTest Text: {test_text}")
print(f"Predicted Sentiment: {predict_sentiment(test_text)}")

Using device: cpu


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers]

Step,Training Loss
1,0.669667
2,0.765120
3,0.703187
4,0.600896
5,0.679406
6,0.610650


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Test Text: Этот продукт превзошел все мои ожидания!
Predicted Sentiment: Positive
